In [ ]:
!pip install -q datasets

In [ ]:
# Standard library imports
import os
import sys
import warnings
import traceback
import gc
import time
import concurrent.futures
from pathlib import Path
from datetime import datetime, timedelta # Added datetime, timedelta

# Type Hinting Imports (consolidated)
from typing import List, Tuple, Optional, Union, Callable, Dict, Any

# Third-party library imports
import numpy as np
import pandas as pd

# Attempt Numba import, proceed without if unavailable
try:
    import numba
    NUMBA_AVAILABLE = True
except ImportError:
    NUMBA_AVAILABLE = False
    # print("Warning: Numba not found. Event mapping might be slower. Install with 'pip install numba'")

# Attempt Hugging Face and Kaggle imports
try:
    from huggingface_hub import HfApi, login, hf_hub_download # Include login and download if used later
    HF_AVAILABLE = True
except ImportError:
    HfApi = None
    login = None
    hf_hub_download = None
    HF_AVAILABLE = False
    # print("Warning: huggingface_hub not found. Hugging Face features disabled. Install with 'pip install huggingface_hub'")

try:
    import kagglehub
    KAGGLE_AVAILABLE = True
except ImportError:
    kagglehub = None # type: ignore
    KAGGLE_AVAILABLE = False
    # print("Warning: kagglehub not found. Kaggle features disabled. Install with 'pip install kagglehub'")

# --- Warnings Configuration ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)


# --- Authentication and API Initialization ---

# Attempt to get secrets from environment variables (recommended for security)
# Assuming UserSecretsClient is available in a specific environment like Kaggle Notebooks
# If not in such an environment, manually set environment variables or use a secure method
try:
    # This block is specific to environments like Kaggle Notebooks
    from kaggle_secrets import UserSecretsClient # Assuming this import is available in the environment
    user_secrets = UserSecretsClient()
    # These will set environment variables internally if not already set
    kaggle_username = user_secrets.get_secret("KAGGLE_USERNAME")
    kaggle_key = user_secrets.get_secret("KAGGLE_KEY")
    hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
    print("Secrets loaded from UserSecretsClient.")

    # Ensure environment variables are set if not automatically done by UserSecretsClient
    if kaggle_username: os.environ["KAGGLE_USERNAME"] = kaggle_username
    if kaggle_key: os.environ["KAGGLE_KEY"] = kaggle_key
    if hf_token: os.environ["HF_TOKEN"] = hf_token

except (ImportError, Exception) as e:
    # This block is for environments where UserSecretsClient is not available
    print(f"Could not load secrets using UserSecretsClient ({e}). Attempting from environment variables...")

    # Try getting from OS environment variables
    kaggle_username = os.environ.get("KAGGLE_USERNAME")
    kaggle_key = os.environ.get("KAGGLE_KEY")
    hf_token = os.environ.get("HF_TOKEN")

    if not kaggle_username or not kaggle_key or not hf_token:
        print("KAGGLE_USERNAME, KAGGLE_KEY, or HF_TOKEN environment variables not found.")
        print("WARNING: Using placeholder values. Replace these with your actual credentials securely.")
        # Fallback (less secure, replace with a secure method like environment variables)
        # Replace these placeholder strings with actual credentials for local testing if needed
        # ENSURE these are NOT committed with actual credentials
        kaggle_username = "YOUR_KAGGLE_USERNAME_PLACEHOLDER"
        kaggle_key = "YOUR_KAGGLE_KEY_PLACEHOLDER"
        hf_token = "YOUR_HF_TOKEN_PLACEHOLDER"

        # Set environment variables even if using placeholders, for consistency
        os.environ["KAGGLE_USERNAME"] = kaggle_username
        os.environ["KAGGLE_KEY"] = kaggle_key
        os.environ["HF_TOKEN"] = hf_token

        print("Using placeholder credentials (less secure).")
    else:
         print("Secrets loaded from environment variables.")


# Initialize Hugging Face API if available and token exists
hf_api: Optional[HfApi] = None
if HF_AVAILABLE and hf_token and hf_token != "YOUR_HF_TOKEN_PLACEHOLDER":
    try:
        # Assuming login is needed even if token is set via env var for some operations
        login(token=hf_token, add_to_git_credential=False) # add_to_git_credential=False usually safer outside notebooks
        hf_api = HfApi()
        print("Hugging Face login successful and API instance created.")
    except Exception as e:
        print(f"Hugging Face login or API initialization failed: {e}")
        hf_api = None # Ensure hf_api is None on failure
else:
    if not HF_AVAILABLE:
         print("Hugging Face API not available (module not imported).")
    elif not hf_token or hf_token == "YOUR_HF_TOKEN_PLACEHOLDER":
         print("Hugging Face token not available or is a placeholder. Skipping login/API init.")


# Note: Kagglehub typically uses environment variables set above directly,
# so no explicit initialization like HfApi() is usually needed for it.
if KAGGLE_AVAILABLE:
    if not kaggle_username or kaggle_username == "YOUR_KAGGLE_USERNAME_PLACEHOLDER" or \
       not kaggle_key or kaggle_key == "YOUR_KAGGLE_KEY_PLACEHOLDER":
           print("Kaggle credentials not available or are placeholders. Kaggle features may not work.")
    else:
           print("Kaggle credentials available.") # Kagglehub uses these automatically
else:
    print("Kagglehub not available (module not imported).")

In [ ]:
# Download latest version
path = kagglehub.dataset_download("mahdiseddigh/dynamic-rhythms-zip")

print("Path to dataset files:", path)

In [ ]:
def find_file_path(
    filename_pattern: str,
    start_dir: Optional[Union[str, Path]] = None,
    return_path_objects: bool = False,
) -> List[Union[str, Path]]:
    """
    Searches for files matching a given pattern recursively within a directory.

    Starts searching from `start_dir` (or the current directory if None) and
    recursively walks down the directory tree to find all files whose names
    match the `filename_pattern`. The pattern can include wildcards like '*'.

    Args:
        filename_pattern (str): The pattern to match against file names
                                (e.g., '*.parquet', 'data_*.csv').
        start_dir (Optional[Union[str, Path]], optional): The directory to start
                                                          the search from. If None,
                                                          starts from the current
                                                          working directory. Defaults
                                                          to None.
        return_path_objects (bool, optional): If True, returns a list of `pathlib.Path`
                                             objects. If False, returns a list of
                                             string representations of the paths.
                                             Defaults to False.

    Returns:
        List[Union[str, Path]]: A list of paths to the found files. Returns an empty
                                list if no files are found or if the start_dir
                                does not exist or is not a directory.
    """
    print(f"Info: find_file_path called for '{filename_pattern}' in '{start_dir}'. (Dummy implementation)")
    if start_dir is None:
        start_path = Path('.') # Use current directory if None
    else:
        start_path = Path(start_dir)

    if not start_path.is_dir():
        print(f"Warning: start_dir '{start_path}' is not a valid directory. Returning empty list.")
        return []

    try:
        # Using Path.rglob for recursive search matching the docstring's description
        found = list(start_path.rglob(filename_pattern))
        # Filter to ensure they are actual files and not directories matching the pattern
        found_files = [p for p in found if p.is_file()]

        if return_path_objects:
            return found_files
        else:
            return [str(p) for p in found_files]
    except Exception as e:
        print(f"Error during recursive search in '{start_path}': {e}")
        traceback.print_exc(limit=1)
        return []

In [ ]:
def make_ts_power_county_yearly(year: int,
                                data_directory: Union[str, Path]
                               ) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    """
    Processes power outage data for a single year from a CSV file.

    Loads the CSV for the specified year, cleans and processes the data,
    extracts a FIPS code to county/state mapping, and aggregates the power
    outage data ('customers_out') by FIPS code and time.

    Args:
        year (int): The year of the data file (e.g., 2021).
        data_directory (Union[str, Path]): The path to the directory containing
                                            the yearly `eaglei_outages_{year}.csv` files.

    Returns:
        Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]: A tuple containing:
            - The aggregated power outage time series DataFrame, indexed by
              ('fips_code', 'time') (Int64, datetime64), with a 'customers_out'
              column (float64 or nullable Int64). Returns None if the file is not
              found, is empty, is missing required columns, or if no valid outage
              data remains after cleaning and aggregation.
            - The FIPS code to county/state mapping DataFrame for this year's data,
              indexed by numeric FIPS code (Int64), with 'county' and 'state'
              columns (str). Returns None if the file is not found, is empty,
              is missing required columns, or if no valid FIPS entries with
              county/state info are found.
    """
    data_dir = Path(data_directory)
    file_name = f"eaglei_outages_{year}.csv"
    file_path = data_dir / file_name
    print(f" [{year}] Processing power data from {file_path}...")

    if not file_path.exists():
        print(f"  [{year}] Warning: File not found: {file_path}. Skipping.")
        return None, None

    try:
        cols_to_read = ['fips_code', 'county', 'state', 'customers_out', 'run_start_time']
        # Specify dtypes for faster loading and less memory usage
        dtype_spec = {
            'fips_code': 'str', # Read as string initially
            'county': 'str',
            'state': 'str',
            'customers_out': 'float64', # Read as float first for potential NaNs
            'run_start_time': 'str'
        }
        df_year = pd.read_csv(
            file_path,
            usecols=lambda c: c in cols_to_read,
            dtype=dtype_spec,
            low_memory=False, # Set to False when specifying dtypes to avoid mixed type inference warnings
        )

        required_cols = ['fips_code', 'county', 'state', 'customers_out', 'run_start_time']
        if not all(col in df_year.columns for col in required_cols):
            print(f"  [{year}] Warning: Required columns missing in {file_path}. Expected {required_cols}, Found {df_year.columns.tolist()}. Skipping.")
            return None, None

        # --- Data Cleaning & Type Conversion ---
        # Drop rows missing FIPS, county, or state (essential for mapping)
        df_year.dropna(subset=['fips_code', 'county', 'state'], inplace=True)
        if df_year.empty:
             print(f"  [{year}] No valid rows after dropping missing FIPS/county/state.")
             return None, None

        # FIPS Map Extraction (handle potential non-numeric FIPS codes gracefully)
        # Extract unique FIPS-county-state mappings
        mapping_part = df_year[['fips_code', 'county', 'state']].drop_duplicates(subset=['fips_code']).copy()
        # Attempt numeric conversion for index
        mapping_part['fips_numeric'] = pd.to_numeric(mapping_part['fips_code'], errors='coerce')
        # Drop rows where FIPS could not be converted to numeric
        mapping_part.dropna(subset=['fips_numeric'], inplace=True)
        # Convert to nullable integer
        mapping_part['fips_numeric'] = mapping_part['fips_numeric'].astype('Int64')

        if not mapping_part.empty:
            # Set the numeric FIPS as index, drop original string fips_code
            mapping_part.set_index('fips_numeric', inplace=True)
            # Ensure index is unique (should be after drop_duplicates, but double-check)
            mapping_part = mapping_part[~mapping_part.index.duplicated(keep='first')].copy()
            # Keep only county and state columns
            mapping_part = mapping_part[['county', 'state']].copy()
        else:
            # If no valid numeric FIPS found for mapping
            mapping_part = None
            print(f"  [{year}] Warning: No valid FIPS entries with county/state info found after cleaning for mapping.")


        # Convert main dataframe columns for aggregation
        df_year['run_start_time'] = pd.to_datetime(df_year['run_start_time'], errors='coerce')
        df_year['customers_out'] = pd.to_numeric(df_year['customers_out'], errors='coerce')
        # Coerce FIPS to numeric Int64, allowing NaNs during conversion before dropping
        df_year['fips_code'] = pd.to_numeric(df_year['fips_code'], errors='coerce').astype('Int64')

        # Drop rows where FIPS, customers_out, or time are NaN after conversion
        df_year.dropna(subset=['fips_code', 'customers_out', 'run_start_time'], inplace=True)

        if df_year.empty:
            print(f"  [{year}] No valid power outage data found after cleaning essential columns.")
            # Return the mapping part if it was successfully created, even if no time series data
            return None, mapping_part

        # --- Aggregation ---
        print(f"  [{year}] Aggregating power data by FIPS and Time...")
        # Group by the cleaned fips_code (Int64) and run_start_time (datetime)
        df_agg = df_year.groupby(['fips_code', 'run_start_time'], observed=True)['customers_out'].sum().reset_index()

        # Convert customers_out to nullable integer if all aggregated values are effectively integers
        if pd.api.types.is_float_dtype(df_agg['customers_out']):
             # Check if all non-NaN values are equal to their rounded value
             if (df_agg['customers_out'].dropna() == df_agg['customers_out'].dropna().round()).all():
                  df_agg['customers_out'] = df_agg['customers_out'].astype('Int64') # Use nullable int


        # Set the MultiIndex
        df_agg.set_index(['fips_code', 'run_start_time'], inplace=True)
        # Rename the index levels
        df_agg.rename_axis(['fips_code', 'time'], inplace=True)
        # Sort the index for efficient operations later
        df_agg.sort_index(inplace=True)

        # Explicitly delete intermediate dataframe to free memory
        del df_year
        gc.collect()

        print(f"  [{year}] Finished power data processing. Found {len(df_agg)} aggregated records.")
        # Return the aggregated time series and the FIPS mapping
        return df_agg, mapping_part

    except pd.errors.EmptyDataError:
        print(f"  [{year}] Warning: File {file_path} is empty. Skipping.")
        return None, None
    except Exception as e:
        print(f"  [{year}] Error processing file {file_path}: {e}")
        traceback.print_exc(limit=1)
        return None, None

In [ ]:
if NUMBA_AVAILABLE:
    @numba.jit(nopython=True, parallel=False, fastmath=True)
    def _numba_map_events_to_grid(
        # Input Event Data (NumPy arrays)
        event_fips_codes: np.ndarray,      # (N_events,) int64 array
        event_type_indices: np.ndarray,    # (N_events,) int64 array (indices into event_type_names)
        event_start_times_ns: np.ndarray,  # (N_events,) int64 array (nanoseconds since epoch)
        event_end_times_ns: np.ndarray,    # (N_events,) int64 array (nanoseconds since epoch)
        # Target Grid Info
        target_fips_codes_unique: np.ndarray, # (N_unique_fips,) int64 array, sorted
        target_time_grid_ns: np.ndarray,   # (N_times,) int64 array, sorted
        target_event_counts: np.ndarray,   # (N_unique_fips * N_times, N_event_types) int16/32 array (modified in-place)
        # Year Boundaries
        year_start_ns: np.int64,
        year_end_ns: np.int64
    ) -> None:
        """
        Numba-accelerated function to map event intervals onto a FIPS/time grid.

        Iterates through each event, clamps its start and end time to the specified
        year boundaries, finds the corresponding FIPS row block in the target grid
        (if the FIPS code exists), and then iterates through the time grid intervals
        that overlap with the clamped event duration. For each overlapping grid interval,
        it increments the count for the specific event type in the target event counts array.
        The target array is modified in-place.

        Args:
            event_fips_codes (np.ndarray): 1D array of integer FIPS codes for each event.
                                           Expected dtype: int64. Shape: (N_events,).
            event_type_indices (np.ndarray): 1D array of integer indices representing
                                             the column index for the event type in the
                                             target counts array. Expected dtype: int64.
                                             Shape: (N_events,).
            event_start_times_ns (np.ndarray): 1D array of event start timestamps in
                                               nanoseconds since the epoch. Expected dtype: int64.
                                               Shape: (N_events,).
            event_end_times_ns (np.ndarray): 1D array of event end timestamps in
                                             nanoseconds since the epoch. Expected dtype: int64.
                                             Shape: (N_events,).
            target_fips_codes_unique (np.ndarray): 1D array of unique, sorted integer
                                                   FIPS codes that define the FIPS dimension
                                                   of the target grid. Expected dtype: int64.
                                                   Shape: (N_unique_fips,).
            target_time_grid_ns (np.ndarray): 1D array of sorted integer timestamps in
                                              nanoseconds that define the time points of the
                                              target grid (e.g., 15-minute intervals).
                                              Expected dtype: int64. Shape: (N_times,).
            target_event_counts (np.ndarray): 2D array representing the target FIPS/time
                                              grid, where rows correspond to (FIPS, Time)
                                              combinations flattened (FIPS blocks concatenated
                                              with time within each block) and columns
                                              correspond to event types. This array is
                                              incremented in-place. Expected dtype: int16 or int32.
                                              Shape: (N_unique_fips * N_times, N_event_types).
            year_start_ns (np.int64): The start timestamp of the processing year in nanoseconds
                                      (inclusive).
            year_end_ns (np.int64): The end timestamp of the processing year in nanoseconds
                                    (exclusive).

        Returns:
            None: The function modifies the `target_event_counts` array in-place.
        """
        n_events = len(event_fips_codes)
        n_times = len(target_time_grid_ns)
        n_unique_fips = len(target_fips_codes_unique)

        for i in range(n_events):
            fips = event_fips_codes[i]
            col_idx = event_type_indices[i] # Direct column index
            start_ns = event_start_times_ns[i]
            end_ns = event_end_times_ns[i]

            # Clamp event to the processing year boundaries [year_start_ns, year_end_ns)
            event_start_clamped = max(start_ns, year_start_ns)
            event_end_clamped = min(end_ns, year_end_ns)

            # Skip if event is outside the year or has zero/negative duration after clamping
            if event_start_clamped >= event_end_clamped:
                continue

            # --- Find FIPS row block in the target grid ---
            # Binary search for FIPS code in the sorted unique FIPS array
            # searchsorted with side='left' finds the index where `fips` would be inserted to maintain order.
            # If `fips` exists, it returns the index of the *first* occurrence.
            fips_row_idx_in_unique = np.searchsorted(target_fips_codes_unique, fips, side='left')

            # Check if FIPS code was found and matches exactly
            # If searchsorted returns n_unique_fips, the value is greater than all elements.
            # If it returns an index < n_unique_fips, we must check if the element at that index is exactly the fips.
            if fips_row_idx_in_unique == n_unique_fips or target_fips_codes_unique[fips_row_idx_in_unique] != fips:
                continue # FIPS code from event not in target grid

            # Calculate the starting row index for this FIPS block in the flattened target_event_counts array
            fips_block_start_row = fips_row_idx_in_unique * n_times

            # --- Find Time Indices in the Target Grid ---
            # Find the *first* grid point whose timestamp is >= event_start_clamped.
            # Using side='left' correctly identifies the start of the interval containing/starting with event_start_clamped.
            # If the event starts *at* a grid point, this finds that point's index.
            start_time_idx = np.searchsorted(target_time_grid_ns, event_start_clamped, side='left')
            # Ensure start_time_idx is within grid bounds (should be >= 0 by searchsorted behavior on sorted array)
            # start_time_idx = max(0, start_time_idx) # Added max(0, ...) just for safety, though searchsorted on sorted >=0 array should handle this.

            # Find the *first* grid point whose timestamp is >= event_end_clamped.
            # Using side='left' ensures that if an event ends exactly *at* a grid point,
            # that grid point's index is returned, and since the loop goes up to *but not including* end_time_idx,
            # the interval ending at event_end_clamped is correctly excluded.
            end_time_idx = np.searchsorted(target_time_grid_ns, event_end_clamped, side='left')


            # --- Increment Counts for Overlapping Intervals ---
            # Iterate through the time grid indices that fall within [event_start_clamped, event_end_clamped)
            # The range is [start_time_idx, end_time_idx).
            # We must also ensure the loop indices are within the bounds of the target_time_grid_ns array [0, n_times).
            # min(end_time_idx, n_times) handles the upper bound correctly.
            # The condition start_time_idx < min(end_time_idx, n_times) ensures we only loop if there's at least one interval.
            # start_time_idx can be == n_times if event starts after last grid point.
            # end_time_idx can be == n_times if event ends after last grid point.
            # If start_time_idx == n_times, the loop range (n_times, min(end_time_idx, n_times)) will be empty, which is correct.
            # If end_time_idx <= start_time_idx, the loop range is also empty, which is correct for zero/negative duration or events falling exactly between grid points.
            for t_idx in range(start_time_idx, min(end_time_idx, n_times)):
                 # Calculate the absolute row index in the flattened target array for this FIPS and time index
                 row_index = fips_block_start_row + t_idx
                 # Check if the row index is valid (should always be if fips was found and t_idx is in range)
                 # Add bounds check for target_event_counts row dimension if necessary, but usually implied by logic
                 # Check if col_idx is valid (should be if event_type_indices are correct)
                 # Add bounds check for target_event_counts column dimension if necessary
                 target_event_counts[row_index, col_idx] += 1

In [ ]:
def _pandas_map_events_to_grid(
    df_events_year: pd.DataFrame,
    target_df: pd.DataFrame, # The pre-created grid DataFrame
    year_start_dt: datetime,
    year_end_dt: datetime, # Inclusive end for filtering
    time_index: pd.DatetimeIndex # The 15-min time index of the target_df
    ) -> None:
    """
    Maps event durations from an event DataFrame onto a FIPS-time grid DataFrame using pandas iteration.

    Iterates through rows of the event DataFrame. For each event, it determines
    which time intervals in the `target_df` (based on its 'time' index level)
    are overlapped by the event's duration (clamped to the processing year).
    It then increments the corresponding event count column in the `target_df`
    for the relevant FIPS code and overlapping time intervals. This function
    modifies `target_df` in-place. This method is generally slower than
    optimized numerical approaches (like Numba).

    Args:
        df_events_year (pd.DataFrame): DataFrame containing event data for a year.
                                       Expected to have columns 'fips_code',
                                       'EVENT_TYPE', 'BEGIN_DATETIME', and 'END_DATETIME'.
        target_df (pd.DataFrame): The target DataFrame representing the FIPS-time grid.
                                  Expected to have a MultiIndex ('fips_code', 'time')
                                  and columns like 'event_count_EVENTTYPE'. This DataFrame
                                  is modified in-place.
        year_start_dt (datetime): The start datetime of the processing year (inclusive).
                                  Used for clamping event durations.
        year_end_dt (datetime): The end datetime of the processing year (inclusive).
                                Used for clamping event durations.
        time_index (pd.DatetimeIndex): The sorted DatetimeIndex corresponding to the
                                       time level of the `target_df` index. Used for
                                       efficient lookups of time interval boundaries.

    Returns:
        None: The function modifies the `target_df` DataFrame in-place and does not
              return a new DataFrame.
    """
    print(f"  [{year_start_dt.year}] Mapping events using pandas loop...")
    processed_count = 0
    total_events = len(df_events_year)

    # Ensure target_df index is sorted for potential loc speedup
    if not target_df.index.is_monotonic_increasing:
         target_df.sort_index(inplace=True)

    # Convert year boundary datetimes to pandas Timestamps once for comparison
    year_start_ts = pd.Timestamp(year_start_dt)
    year_end_ts = pd.Timestamp(year_end_dt)


    for _, row in df_events_year.iterrows():
        fips = row['fips_code']
        event_type_col = f'event_count_{row["EVENT_TYPE"]}'

        # Check if FIPS and column exist in the target grid *before* proceeding
        # Check FIPS in the first level of the index
        if fips not in target_df.index.get_level_values('fips_code') or event_type_col not in target_df.columns:
            processed_count += 1
            continue # Skip if FIPS or event type column doesn't exist

        # Get event start and end times as pandas Timestamps
        event_start_ts = pd.Timestamp(row['BEGIN_DATETIME'])
        event_end_ts = pd.Timestamp(row['END_DATETIME'])

        # Clamp event to the year boundary using Timestamps
        event_start_clamped = max(event_start_ts, year_start_ts)
        # Use inclusive end date for filtering/clamping, but remember intervals are exclusive of end point
        event_end_clamped = min(event_end_ts, year_end_ts)

        # Skip if event has zero or negative duration after clamping
        if event_start_clamped >= event_end_clamped:
            processed_count += 1
            continue

        # Rounding logic: floor event start to the nearest 15-min interval start
        event_start_rounded = event_start_clamped.floor('15min')
        # Ceil event end to the nearest 15-min interval end. Add a tiny delta before ceiling
        # to ensure events ending exactly on a boundary are included in the *last* interval.
        event_end_ceil = (event_end_clamped + timedelta(microseconds=1)).ceil('15min')

        # Find index positions in the pre-defined time_index
        # searchsorted with side='left' finds the index where value should be inserted to maintain order.
        # For start_idx_pos, side='left' finds the first time >= event_start_rounded.
        # For end_idx_pos, side='left' finds the first time >= event_end_ceil.
        start_idx_pos = time_index.searchsorted(event_start_rounded, side='left')
        end_idx_pos = time_index.searchsorted(event_end_ceil, side='left') # The slice end is exclusive


        # Ensure the slice is valid and non-empty before attempting to index
        # start_idx_pos must be less than the total number of time points to be a valid start.
        # end_idx_pos must be > start_idx_pos for a valid, non-empty slice.
        # min(end_idx_pos, len(time_index)) ensures we don't go past the end of time_index.
        actual_end_idx_pos = min(end_idx_pos, len(time_index))

        if start_idx_pos < actual_end_idx_pos:
            # Define the slice for .loc using the actual time index values
            # Get the specific time stamps from the time_index based on the found positions
            times_to_increment = time_index[start_idx_pos : actual_end_idx_pos]
            idx_slice = pd.IndexSlice[fips, times_to_increment]
            try:
                # Increment using .loc
                # This operation can be slow if the slice is large or done repeatedly
                target_df.loc[idx_slice, event_type_col] += 1
            except KeyError:
                 # This can happen if the FIPS or specific time values in the slice
                 # are not present in the target_df's index after the initial checks.
                 # This might indicate an issue with index alignment or missing data points.
                 # print(f"    Warning: KeyError during increment for FIPS {fips}, times range from {times_to_increment.min()} to {times_to_increment.max()}, event {event_type_col}.")
                 pass # Ignore the error and continue to the next event
            except Exception as e_inc:
                 # Catch any other unexpected errors during the increment
                 print(f"    Error incrementing event count for FIPS {fips}, event {event_type_col}: {e_inc}")


        processed_count += 1
        if processed_count % 50000 == 0:
             # Use year from year_start_dt for printing
             print(f"    [{year_start_dt.year}] Processed {processed_count}/{total_events} events...")

    # Final print statement using year from year_start_dt
    print(f"  [{year_start_dt.year}] Finished pandas event mapping.")

In [ ]:
def make_ts_events_county_yearly(year: int,
                                 noaa_df_year: pd.DataFrame # Pre-filtered for the year
                                ) -> Optional[pd.DataFrame]:
    """
    Creates a 15-minute time series DataFrame of NOAA event counts for a given year.
    Uses Numba for mapping if available, otherwise falls back to pandas.

    Args:
        year: The year to process.
        noaa_df_year: DataFrame containing NOAA events already filtered for the year
                      and with required columns ('fips_code', 'EVENT_TYPE',
                      'BEGIN_DATETIME', 'END_DATETIME').

    Returns:
        Optional[pd.DataFrame]: DataFrame with MultiIndex (fips_code, time)
                                 and columns for event counts (event_count_TYPE),
                                 or None if processing fails.
    """
    print(f" [{year}] Processing NOAA event data...")
    if noaa_df_year.empty:
        print(f"  [{year}] No NOAA event data provided. Returning empty events DataFrame.")
        # Return structure expected by merge step
        return pd.DataFrame(index=pd.MultiIndex(levels=[[],[]], codes=[[],[]], names=['fips_code', 'time']))

    start_time = time.time()

    # --- Define Time Range and Grid ---
    year_start_dt = datetime(year, 1, 1, 0, 0, 0)
    # Inclusive end for filtering comparison, exclusive used in range/grid generation
    year_end_filter_dt = datetime(year, 12, 31, 23, 59, 59)
    # Grid should end at the last 15-min interval *of* the year
    year_end_grid_dt = datetime(year, 12, 31, 23, 45, 0)

    # Ensure required columns are present and clean data
    required_cols = ['fips_code', 'EVENT_TYPE', 'BEGIN_DATETIME', 'END_DATETIME']
    if not all(col in noaa_df_year.columns for col in required_cols):
         print(f"  [{year}] Error: NOAA data missing required columns: {required_cols}")
         return None
    noaa_df_year.dropna(subset=required_cols, inplace=True)
    # Ensure fips is Int64
    noaa_df_year['fips_code'] = pd.to_numeric(noaa_df_year['fips_code'], errors='coerce').astype('Int64')
    noaa_df_year.dropna(subset=['fips_code'], inplace=True) # Drop if FIPS became NA
    # Convert EVENT_TYPE to category for efficiency
    noaa_df_year['EVENT_TYPE'] = noaa_df_year['EVENT_TYPE'].astype('category')


    if noaa_df_year.empty:
        print(f"  [{year}] No valid NOAA events remain after cleaning.")
        return pd.DataFrame(index=pd.MultiIndex(levels=[[],[]], codes=[[],[]], names=['fips_code', 'time']))

    # --- Create Target DataFrame Structure ---
    unique_fips = sorted(noaa_df_year['fips_code'].unique())
    unique_event_types = sorted(noaa_df_year['EVENT_TYPE'].cat.categories.tolist())
    print(f"  [{year}] Found {len(unique_fips)} FIPS codes and {len(unique_event_types)} unique event types.")

    if not unique_fips:
        print(f"  [{year}] No valid FIPS codes found. Skipping event processing.")
        return pd.DataFrame(index=pd.MultiIndex(levels=[[],[]], codes=[[],[]], names=['fips_code', 'time']))

    time_index = pd.date_range(start=year_start_dt, end=year_end_grid_dt, freq='15min', name='time')
    multi_index = pd.MultiIndex.from_product([unique_fips, time_index], names=['fips_code', 'time'])

    # Use int16 for counts if possible, check max potential count if necessary
    # If a single interval could have > 32767 events of one type, use int32
    dtype_event_count = 'int16'
    event_columns = [f'event_count_{etype}' for etype in unique_event_types]
    # Create the DataFrame filled with zeros
    target_df = pd.DataFrame(0, index=multi_index, columns=event_columns, dtype=dtype_event_count)
    print(f"  [{year}] Created target event grid shape: {target_df.shape}")

    # --- Map Events to Grid (Use Numba if possible) ---
    if NUMBA_AVAILABLE:
        try:
            print(f"  [{year}] Preparing data for Numba acceleration...")
            # Prepare NumPy arrays for Numba function
            event_fips_np = noaa_df_year['fips_code'].to_numpy(dtype=np.int64)
            # Map event types to integer column indices
            event_type_map = {etype: i for i, etype in enumerate(target_df.columns)}
            event_type_indices_np = noaa_df_year['EVENT_TYPE'].cat.codes.map(
                lambda code: event_type_map.get(f'event_count_{noaa_df_year["EVENT_TYPE"].cat.categories[code]}', -1)
            ).to_numpy(dtype=np.int64) # Map category codes to column index

            # Convert times to nanoseconds (int64)
            event_start_ns = noaa_df_year['BEGIN_DATETIME'].values.astype(np.int64)
            event_end_ns = noaa_df_year['END_DATETIME'].values.astype(np.int64)

            # Target grid info as NumPy arrays
            target_fips_unique_np = target_df.index.levels[0].to_numpy(dtype=np.int64) # Already sorted unique FIPS
            target_time_grid_ns_np = target_df.index.levels[1].values.astype(np.int64) # Time grid
            target_event_counts_np = target_df.values # Get the underlying array (Fortran order from_product)
            # Ensure C-contiguous order for potentially better Numba performance if needed
            if not target_event_counts_np.flags['C_CONTIGUOUS']:
                 target_event_counts_np = np.ascontiguousarray(target_event_counts_np)
                 # NOTE: If made contiguous, it's a copy. We need to update the original DF later.
                 # For now, assume modifying the original view works or Numba handles Fortran order.

            year_start_ns_np = np.datetime64(year_start_dt).astype(np.int64)
            # Use exclusive end for Numba logic (consistent with pandas slicing)
            year_end_ns_np = np.datetime64(year_end_filter_dt + timedelta(microseconds=1)).astype(np.int64)

            print(f"  [{year}] Mapping events using Numba...")
            # Call the JIT compiled function
            _numba_map_events_to_grid(
                event_fips_np,
                event_type_indices_np,
                event_start_ns,
                event_end_ns,
                target_fips_unique_np,
                target_time_grid_ns_np,
                target_event_counts_np, # Pass the array to be modified
                year_start_ns_np,
                year_end_ns_np
            )
            # If target_event_counts_np was a copy, update the DataFrame:
            # if not target_df.values.flags['C_CONTIGUOUS']: # Check if we made a copy
            #    target_df[:] = target_event_counts_np # Assign values back

            print(f"  [{year}] Finished Numba event mapping.")

        except Exception as e_numba:
            print(f"  [{year}] Error during Numba execution: {e_numba}")
            print(f"  [{year}] Falling back to pandas mapping loop...")
            traceback.print_exc(limit=1)
            # Fallback to pandas if Numba fails
            _pandas_map_events_to_grid(noaa_df_year, target_df, year_start_dt, year_end_filter_dt, time_index)

    else: # Numba not available
        _pandas_map_events_to_grid(noaa_df_year, target_df, year_start_dt, year_end_filter_dt, time_index)


    # Optional: Remove rows where all event counts are zero AFTER processing
    # print(f"  [{year}] Dropping rows with no events...")
    # target_df = target_df[target_df.any(axis=1)]
    # print(f"  [{year}] Final event DataFrame shape after dropping zeros: {target_df.shape}")

    processing_time = time.time() - start_time
    print(f"  [{year}] Event processing finished in {processing_time:.2f} seconds.")
    return target_df

In [ ]:
def upload_yearly_file_to_hf(local_file_path: Path,
                             repo_id: str,
                             hf_api_instance: Optional[HfApi],
                             repo_type: str = "dataset") -> bool:
    """Uploads a single file to Hugging Face Hub."""
    if not HF_AVAILABLE or hf_api_instance is None:
        # print(f"  Skipping HF upload for {local_file_path.name}: API client not available.")
        return False # Indicate upload was not attempted/successful

    if not local_file_path.exists():
        print(f"  Error: Local file {local_file_path} not found. Cannot upload.")
        return False

    target_path_in_repo = local_file_path.name
    commit_msg = f"Add/Update {target_path_in_repo}"
    print(f"  Uploading {local_file_path.name} to HF Hub repo {repo_id}...")

    try:
        hf_api_instance.upload_file(
            path_or_fileobj=str(local_file_path),
            path_in_repo=target_path_in_repo,
            repo_id=repo_id,
            repo_type=repo_type,
            commit_message=commit_msg,
            # Consider adding retries if needed
            # retry_endpoint_failed=...
        )
        print(f"  Successfully uploaded {target_path_in_repo} to {repo_id}.")
        return True
    except Exception as e:
        print(f"  Error uploading {local_file_path.name} to Hugging Face Hub: {e}")
        traceback.print_exc(limit=1)
        return False

In [ ]:
def process_year(year: int,
                 power_data_dir: Path,
                 noaa_df_full: pd.DataFrame, # Pass the full loaded NOAA df
                 output_dir: Path,
                 hf_repo_id: str,
                 hf_api_instance: Optional[HfApi], # Pass API instance
                 delete_local_after_upload: bool = False # Option to save disk space
                 ) -> Tuple[int, bool, Optional[pd.DataFrame], bool]:
    """
    Processes power and event data for a single year, merges them,
    saves the result locally, uploads to Hugging Face Hub (optional),
    and performs memory cleanup. Designed for parallel execution.

    Returns:
        Tuple[int, bool, Optional[pd.DataFrame], bool]:
            - Year processed.
            - Overall processing & local save success flag for the year.
            - DataFrame containing the FIPS map for this year (or None).
            - Success flag for the Hugging Face upload step.
    """
    print(f"\n--- Starting Year: {year} ---")
    start_process_time = time.time()
    processing_success = False
    hf_upload_success = False
    fips_map_yearly_agg = None # Initialize local FIPS map for this year

    # --- 1. Process Power Data ---
    df_power_yearly, fips_map_yearly = make_ts_power_county_yearly(year, power_data_dir)

    # Store the FIPS map found this year (keep original fips_code str if needed)
    if fips_map_yearly is not None and not fips_map_yearly.empty:
        fips_map_yearly_agg = fips_map_yearly # Assign to the variable we'll return
        print(f"  [{year}] Found {len(fips_map_yearly_agg)} FIPS entries in power data.")

    if df_power_yearly is None or df_power_yearly.empty:
        print(f"  [{year}] No power data. Skipping further processing for this year.")
        del fips_map_yearly
        gc.collect()
        return year, processing_success, fips_map_yearly_agg, hf_upload_success

    # --- 2. Filter and Process Event Data ---
    print(f"  [{year}] Filtering NOAA data...")
    year_start_dt = datetime(year, 1, 1, 0, 0, 0)
    year_end_dt = datetime(year, 12, 31, 23, 59, 59)

    # Filter the *full* df passed as argument using boolean indexing
    noaa_df_year_filtered = noaa_df_full[
        (noaa_df_full['END_DATETIME'] >= year_start_dt) &
        (noaa_df_full['BEGIN_DATETIME'] <= year_end_dt)
    ].copy() # Use .copy() crucial to avoid modifying the shared full df view

    if noaa_df_year_filtered.empty:
        print(f"  [{year}] No overlapping NOAA events found.")
        # Create an empty events DataFrame with standard index names for merging
        df_events_yearly = pd.DataFrame(index=pd.MultiIndex(levels=[[],[]], codes=[[],[]], names=['fips_code', 'time']))
    else:
        # Process the filtered events
        df_events_yearly = make_ts_events_county_yearly(year, noaa_df_year_filtered)

    del noaa_df_year_filtered # Free memory of the filtered slice
    gc.collect()

    if df_events_yearly is None: # Check if event processing function indicated failure
        print(f"  [{year}] Event processing failed. Cannot proceed.")
        del df_power_yearly
        if 'fips_map_yearly' in locals(): del fips_map_yearly # Clean up power map if it existed
        gc.collect()
        return year, processing_success, fips_map_yearly_agg, hf_upload_success

    # --- 3. Merge Data ---
    print(f"  [{year}] Merging power and event data (left join)...")
    # Ensure indices are compatible before merge
    if not df_power_yearly.index.is_monotonic_increasing: df_power_yearly.sort_index(inplace=True)
    if not df_events_yearly.index.is_monotonic_increasing: df_events_yearly.sort_index(inplace=True)

    df_combined_yearly = pd.merge(
        df_power_yearly, df_events_yearly,
        left_index=True, right_index=True,
        how='left' # Keep all power outage rows, add event counts where they align
    )
    print(f"  [{year}] Merged DataFrame shape: {df_combined_yearly.shape}")

    del df_power_yearly
    del df_events_yearly
    gc.collect()

    # --- 4. Clean and Finalize Merged Data ---
    print(f"  [{year}] Finalizing merged data (filling NaNs, adding county/state)...")
    event_cols = [col for col in df_combined_yearly.columns if col.startswith('event_count_')]
    if event_cols:
        # Fill NaNs introduced by the left merge (where events didn't align with power outages)
        # Use the determined event count dtype (e.g., 'int16')
        event_fill_type = 'int16' # Default if original dtype cannot be determined
        try:
             # Infer from a non-empty column if possible (less reliable)
             # Or rely on the dtype used when creating target_df
             pass
        except: pass
        df_combined_yearly[event_cols] = df_combined_yearly[event_cols].fillna(0).astype(event_fill_type)


    # Add county/state using the *local* FIPS map generated for this year
    if fips_map_yearly_agg is not None and not fips_map_yearly_agg.empty:
        try:
            # Ensure index types match before merge
            map_index_dtype = fips_map_yearly_agg.index.dtype
            data_fips_dtype = df_combined_yearly.index.get_level_values('fips_code').dtype
            if map_index_dtype != data_fips_dtype:
                # print(f"   [{year}] Adjusting FIPS map index dtype ({map_index_dtype}) to match data FIPS ({data_fips_dtype})")
                # Be cautious changing index type, ensure it's the right conversion
                try:
                     fips_map_yearly_agg.index = fips_map_yearly_agg.index.astype(data_fips_dtype)
                except Exception as e_dtype:
                     print(f"   [{year}] Warning: Could not align FIPS map index dtype: {e_dtype}. Skipping county/state merge.")
                     fips_map_yearly_agg = None # Prevent merge attempt

            if fips_map_yearly_agg is not None: # Check again after potential dtype adjustment failure
                 # Select only county/state, avoid duplicate columns if fips_code was kept in map
                 cols_to_merge = ['county', 'state']
                 if 'fips_code' in fips_map_yearly_agg.columns:
                      fips_map_to_merge = fips_map_yearly_agg[cols_to_merge]
                 else:
                      fips_map_to_merge = fips_map_yearly_agg[cols_to_merge]

                 df_combined_yearly = df_combined_yearly.merge(
                     fips_map_to_merge,
                     left_on='fips_code', right_index=True,
                     how='left'
                 )
                 # Reorder columns: county, state first
                 meta_cols = ['county', 'state']
                 data_cols = [col for col in df_combined_yearly.columns if col not in meta_cols]
                 df_combined_yearly = df_combined_yearly[meta_cols + data_cols]

        except Exception as merge_err:
             print(f"   [{year}] Error merging FIPS map: {merge_err}. County/State columns might be missing or incomplete.")
             traceback.print_exc(limit=1)
             # Add empty columns if merge fails completely
             if 'county' not in df_combined_yearly.columns: df_combined_yearly['county'] = pd.NA
             if 'state' not in df_combined_yearly.columns: df_combined_yearly['state'] = pd.NA
    else:
        # Add empty columns if no FIPS map was generated or merge failed
        if 'county' not in df_combined_yearly.columns: df_combined_yearly['county'] = pd.NA
        if 'state' not in df_combined_yearly.columns: df_combined_yearly['state'] = pd.NA

    # Ensure final DataFrame is sorted
    if not df_combined_yearly.index.is_monotonic_increasing:
        df_combined_yearly.sort_index(inplace=True)

    # --- 5. Save Locally ---
    output_file = output_dir / f'combined_county_15min_{year}.parquet'
    print(f"  [{year}] Saving combined data to {output_file}...")
    try:
        # Use pyarrow engine if available, specify compression for space
        parquet_engine = 'pyarrow' if 'pyarrow' in sys.modules else 'fastparquet'
        df_combined_yearly.to_parquet(output_file, index=True, engine=parquet_engine, compression='snappy')
        print(f"  [{year}] Successfully saved {output_file}")
        processing_success = True # Mark as success ONLY if save works
    except Exception as e:
        print(f"  [{year}] Error saving file {output_file}: {e}")
        traceback.print_exc(limit=1)
        # If saving fails, we cannot upload. processing_success remains False.

    # --- 6. Upload to Hugging Face ---
    if processing_success and hf_api_instance: # Only attempt upload if local save worked AND api instance exists
        hf_upload_success = upload_yearly_file_to_hf(
            local_file_path=output_file,
            repo_id=hf_repo_id,
            hf_api_instance=hf_api_instance
        )
        # Optional: Delete local file after successful upload
        if hf_upload_success and delete_local_after_upload:
            try:
                output_file.unlink()
                print(f"  [{year}] Deleted local file {output_file} after successful HF upload.")
            except OSError as e:
                print(f"  [{year}] Warning: Could not delete local file {output_file}: {e}")
    elif processing_success and not hf_api_instance:
         # print(f"  [{year}] Skipping HF upload because API instance is not available.")
         pass # Silently skip if no API

    # --- Cleanup for the year ---
    del df_combined_yearly
    # fips_map_yearly is already deleted if power processing failed, otherwise kept in fips_map_yearly_agg
    # Only explicitly delete the one returned if power processing *succeeded* but we don't need it anymore
    if 'fips_map_yearly' in locals() and fips_map_yearly is not fips_map_yearly_agg:
         del fips_map_yearly
    gc.collect()

    end_process_time = time.time()
    print(f"--- Finished Year: {year} in {end_process_time - start_process_time:.2f} seconds ---")

    # Return year, processing status, the FIPS map *for this year*, and HF upload status
    return year, processing_success, fips_map_yearly_agg, hf_upload_success


In [ ]:
def finalize_data(output_dir: Path,
                  all_fips_maps: List[Optional[pd.DataFrame]], # List of maps from each year
                  hf_repo_id: str,
                  hf_api_instance: Optional[HfApi],
                  final_output_name_base: str = "final_combined_county"):
    """
    Aggregates FIPS maps from all processed years, saves the unique mapping
    locally, and uploads it to HF Hub.
    """
    print("\n--- Finalizing Data: Aggregating and Saving FIPS Map ---")

    # --- Aggregate FIPS Maps ---
    aggregated_fips_map = None
    valid_maps = [m for m in all_fips_maps if m is not None and not m.empty]

    if valid_maps:
        print(f"Aggregating FIPS maps from {len(valid_maps)} year(s)...")
        try:
            aggregated_fips_map = pd.concat(valid_maps)
            # Ensure index has a name ('fips_numeric' based on make_ts_power_county_yearly)
            if aggregated_fips_map.index.name is None:
                 aggregated_fips_map.index.name = 'fips_numeric'

            # Remove duplicate FIPS codes, keeping the first encountered county/state pair
            aggregated_fips_map = aggregated_fips_map[~aggregated_fips_map.index.duplicated(keep='first')]
            aggregated_fips_map.sort_index(inplace=True)
            print(f"Aggregated FIPS map contains {len(aggregated_fips_map)} unique entries.")
        except Exception as e:
             print(f"Error concatenating FIPS maps: {e}")
             traceback.print_exc(limit=1)
             aggregated_fips_map = None # Ensure it's None if concat fails
    else:
        print("No valid FIPS maps were collected from yearly processing.")
        # Create an empty DataFrame with expected structure for consistency downstream
        aggregated_fips_map = pd.DataFrame(columns=['fips_code', 'county', 'state'])
        # aggregated_fips_map.set_index('fips_numeric', inplace=True) # Or define schema more robustly


    # --- Save Aggregated FIPS Map Locally ---
    fips_map_file = output_dir / f"{final_output_name_base}_fips_map.csv"
    map_saved_locally = False
    if aggregated_fips_map is not None and not aggregated_fips_map.empty:
        try:
            print(f"Saving aggregated FIPS map to {fips_map_file}...")
            aggregated_fips_map.to_csv(fips_map_file, index=True) # Index=True saves the FIPS code
            print("... Aggregated FIPS map saved locally.")
            map_saved_locally = True
        except Exception as e:
            print(f"Error saving aggregated FIPS map locally: {e}")
            traceback.print_exc(limit=1)
    else:
        print("Skipping local save of FIPS map (empty or aggregation failed).")

    # --- Upload Aggregated FIPS Map to Hugging Face Hub ---
    if map_saved_locally and hf_api_instance:
        print("\nUploading aggregated FIPS map to Hugging Face Hub...")
        upload_yearly_file_to_hf(
            local_file_path=fips_map_file,
            repo_id=hf_repo_id,
            hf_api_instance=hf_api_instance
        )
    elif map_saved_locally and not hf_api_instance:
         # print("\nSkipping FIPS map upload to Hugging Face Hub (API client not available).")
         pass
    elif not map_saved_locally:
        print("\nSkipping FIPS map upload to Hugging Face Hub (local save failed or map empty).")

    print("--- Finalization Complete ---")

In [ ]:
if __name__ == "__main__":
    main_start_time = time.time()

    # --- Configuration ---
    # Adjust years as needed
    start_year = 2015 # Don't use 2014! it is smaller and it makes the parallel process fail
    end_year = 2023

    # Determine worker count - start conservatively, adjust based on memory/CPU load
    num_cores = os.cpu_count() or 1
    # Leave a couple of cores for OS/other tasks, cap at a reasonable number (e.g., 32 or 64)
    # Too many workers might lead to memory exhaustion or diminishing returns due to overhead
    max_workers = min(max(1, num_cores - 2), 64) # Example: cap at 64, leave 2 cores free
    max_workers = 8
    print(f"System CPU count: {num_cores}. Using max_workers: {max_workers}")

    # --- Paths ---
    # !! IMPORTANT: Update these paths to match your environment !!
    try:
        # Assume Kaggle environment paths - adjust if running elsewhere
        base_data_dir = Path('/root/.cache/kagglehub/datasets/mahdiseddigh/dynamic-rhythms-zip/versions/1/data/')
        power_dir = base_data_dir / 'eaglei_data/'
        noaa_file = base_data_dir / 'NOAA_StormEvents/StormEvents_2014_2024.csv'
        output_base_dir = Path('/content/') # Or '/kaggle/working/' in Kaggle notebooks
    except Exception as e:
        print(f"Error setting up initial paths: {e}")
        print("Please verify the paths for power_dir, noaa_file, and output_base_dir.")
        sys.exit(1)

    output_subdir_name = 'dynamic_rhythms_yearly_output_parallel_final'
    output_dir = output_base_dir / output_subdir_name

    # --- Hugging Face Configuration ---
    # !! IMPORTANT: Set your HF username and desired repo name !!
    hf_username = "mhdaw" # Replace with your HF username
    hf_repo_name = "Dynamic-Rhythms-merged-15min-yearly-parquet-colab"
    hf_repo_id = f"{hf_username}/{hf_repo_name}"
    # Option: Delete local .parquet files after successful HF upload to save space?
    delete_local_after_hf_upload = False

    # --- Kaggle Dataset Configuration ---
    # !! IMPORTANT: Set your Kaggle username !!
    kaggle_username = os.environ.get("KAGGLE_USERNAME") # Best practice: use environment variable
    if not kaggle_username:
        print("Warning: KAGGLE_USERNAME environment variable not set. Using placeholder.")
        kaggle_username = "your_kaggle_username" # Fallback placeholder

    dataset_name = "dynamic-rhythms-yearly-15min-parquet-colab" # Use a descriptive name
    kaggle_handle = f"{kaggle_username}/{dataset_name}"


    # --- Setup Hugging Face API Client ---
    hf_api = None
    if HF_AVAILABLE:
        try:
            print("\nAttempting Hugging Face authentication...")
            # Prioritize token from environment variable
            hf_token = os.environ.get("HF_TOKEN")
            if hf_token:
                hf_api = HfApi(token=hf_token)
                print("  Authenticated using HF_TOKEN environment variable.")
            else:
                # Try default login (e.g., cached token, interactive login if available)
                # You might need notebook_login() or hf_hub_login() depending on environment
                # For non-interactive: ensure token is cached via `huggingface-cli login`
                # Placeholder for attempting default login methods
                 hf_api = HfApi() # Tries to find token in cache/env
                 print("  Attempting authentication using cached token or default methods.")

            # Verify authentication works
            user_info = hf_api.whoami()
            print(f"  Hugging Face authentication successful for user: {user_info.get('name')}")

            # Create HF repo if it doesn't exist
            print(f"  Ensuring Hugging Face dataset repository exists: {hf_repo_id}")
            hf_api.create_repo(repo_id=hf_repo_id, repo_type="dataset", exist_ok=True)
            print(f"  HF repository {hf_repo_id} ensured.")

        except Exception as e:
            print(f"  Hugging Face authentication or repo creation failed: {e}")
            print("  Hugging Face uploads will be skipped.")
            hf_api = None # Ensure hf_api is None if setup fails
    else:
        print("\nHugging Face Hub library not available. Skipping HF setup and uploads.")

    # --- Print Configuration ---
    print("\n--- Starting Parallel Yearly County-Level Data Processing ---")
    print(f"Year Range: {start_year} to {end_year}")
    print(f"Power Data Source: {power_dir}")
    print(f"NOAA Data Source: {noaa_file}")
    print(f"Output Directory: {output_dir}")
    print(f"Using Numba: {NUMBA_AVAILABLE}")
    print(f"Hugging Face Repo: {hf_repo_id if HF_AVAILABLE else 'N/A'}")
    print(f"Delete Local Parquet after HF Upload: {delete_local_after_hf_upload}")
    print(f"Kaggle Dataset Handle: {kaggle_handle if KAGGLE_AVAILABLE else 'N/A'}")
    print("-" * 60)

    # --- Create Output Directory ---
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Output directory created/exists: {output_dir}")
    except Exception as e:
        print(f"Error creating output directory {output_dir}: {e}")
        sys.exit(1)


    # --- Pre-load and Prepare NOAA Data ONCE ---
    # This is memory intensive but avoids reading the large CSV in each process
    full_noaa_df = None
    print("\nLoading and pre-processing full NOAA storm event data...")
    noaa_load_start = time.time()
    try:
        # Define columns and dtypes for efficient loading
        noaa_cols_to_load = [
            'BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME',
            'END_YEARMONTH', 'END_DAY', 'END_TIME',
            'STATE_FIPS', 'CZ_FIPS', 'EVENT_TYPE' # Core data needed
            # Add CZ_TIMEZONE if needed for more precise time alignment (requires more complex logic)
        ]
        # Use efficient dtypes
        dtype_map_noaa = {
            'BEGIN_YEARMONTH': 'int32', 'BEGIN_DAY': 'int8', 'BEGIN_TIME': 'int16',
            'END_YEARMONTH': 'int32', 'END_DAY': 'int8', 'END_TIME': 'int16',
            'STATE_FIPS': 'str', 'CZ_FIPS': 'str', # Read as string for robust FIPS handling
            'EVENT_TYPE': 'category' # Very important for memory and speed
        }
        full_noaa_df = pd.read_csv(
            noaa_file,
            usecols=noaa_cols_to_load,
            dtype=dtype_map_noaa,
            low_memory=False # Set low_memory=False when specifying dtypes
        )
        print(f"  Loaded NOAA data shape: {full_noaa_df.shape}")
        print(f"  Initial memory usage: {full_noaa_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")


        # --- Pre-calculate FIPS codes and Datetimes ---
        print("  Pre-calculating FIPS codes...")
        # Combine STATE_FIPS and CZ_FIPS, ensuring correct padding
        full_noaa_df.dropna(subset=['STATE_FIPS', 'CZ_FIPS'], inplace=True)
        # Pad FIPS components correctly before combining
        full_noaa_df['STATE_FIPS_PAD'] = full_noaa_df['STATE_FIPS'].astype(str).str.zfill(2)
        full_noaa_df['CZ_FIPS_PAD'] = full_noaa_df['CZ_FIPS'].astype(str).str.zfill(3)
        full_noaa_df['fips_code_str'] = full_noaa_df['STATE_FIPS_PAD'] + full_noaa_df['CZ_FIPS_PAD']
        # Convert to numeric Int64, handling potential errors
        full_noaa_df['fips_code'] = pd.to_numeric(full_noaa_df['fips_code_str'], errors='coerce').astype('Int64')

        # Drop intermediate columns if not needed
        full_noaa_df.drop(columns=['STATE_FIPS', 'CZ_FIPS', 'STATE_FIPS_PAD', 'CZ_FIPS_PAD', 'fips_code_str'], inplace=True)


        print("  Pre-calculating Datetimes (this may take a moment)...")
        # Vectorized datetime creation (usually faster than .apply)
        def create_datetime_vectorized(df, year_col, day_col, time_col):
            time_str = df[time_col].astype(int).astype(str).str.zfill(4)
            # Handle potential invalid date components before conversion
            date_str = (df[year_col].astype(int).astype(str) +
                        df[day_col].astype(int).astype(str).str.zfill(2) +
                        time_str)
            # errors='coerce' will turn invalid dates/times into NaT
            return pd.to_datetime(date_str, format='%Y%m%d%H%M', errors='coerce')

        full_noaa_df['BEGIN_DATETIME'] = create_datetime_vectorized(full_noaa_df, 'BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME')
        full_noaa_df['END_DATETIME'] = create_datetime_vectorized(full_noaa_df, 'END_YEARMONTH', 'END_DAY', 'END_TIME')

        # Drop original date/time component columns
        full_noaa_df.drop(columns=['BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME',
                                   'END_YEARMONTH', 'END_DAY', 'END_TIME'], inplace=True)


        # --- Final Cleaning and Filtering ---
        cols_to_check_final = ['fips_code', 'BEGIN_DATETIME', 'END_DATETIME', 'EVENT_TYPE']
        initial_rows = len(full_noaa_df)
        full_noaa_df.dropna(subset=cols_to_check_final, inplace=True)
        # Ensure end time is not before start time
        full_noaa_df = full_noaa_df[full_noaa_df['END_DATETIME'] >= full_noaa_df['BEGIN_DATETIME']]
        dropped_rows = initial_rows - len(full_noaa_df)

        print(f"  Dropped {dropped_rows} rows from NOAA data due to missing/invalid FIPS/DateTime/EventType.")
        print(f"  Final pre-processed NOAA data shape: {full_noaa_df.shape}")
        # Optimize memory usage further
        # Convert object columns to category if appropriate (already done for EVENT_TYPE)
        gc.collect() # Collect garbage after drops
        print(f"  Final memory usage: {full_noaa_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
        noaa_load_time = time.time() - noaa_load_start
        print(f"NOAA data loaded and pre-processed in {noaa_load_time:.2f} seconds.")


    except FileNotFoundError:
        print(f"Error: NOAA CSV file not found at {noaa_file}. Exiting.")
        sys.exit(1)
    except Exception as e:
        print(f"Error loading or pre-processing NOAA CSV file {noaa_file}: {e}")
        traceback.print_exc()
        sys.exit(1)
    if full_noaa_df is None or full_noaa_df.empty:
         print("Error: NOAA data processing resulted in an empty DataFrame. Exiting.")
         sys.exit(1)


    # --- Initialize result containers ---
    futures = []
    results_fips_maps_list = [] # Store the FIPS maps returned by each process
    processed_years_summary = {} # Store status per year {year: (proc_success, hf_success)}

    # --- Submit tasks to ProcessPoolExecutor ---
    # Use try...finally to ensure executor shutdown
    executor = concurrent.futures.ProcessPoolExecutor(max_workers=max_workers)
    try:
        print(f"\nSubmitting tasks for years {start_year}-{end_year} to {max_workers} workers...")
        for current_year in range(start_year, end_year + 1):
            future = executor.submit(
                process_year, # The function to execute
                # --- Arguments for process_year ---
                year=current_year,
                power_data_dir=power_dir,
                noaa_df_full=full_noaa_df, # Pass the whole pre-processed df (read-only access)
                output_dir=output_dir,
                hf_repo_id=hf_repo_id,
                hf_api_instance=hf_api, # Pass the logged-in API instance
                delete_local_after_upload=delete_local_after_hf_upload
            )
            futures.append(future)

        print("All tasks submitted. Waiting for results...")
        # --- Collect results as they complete ---
        # Use tqdm here for a progress bar if installed: from tqdm import tqdm
        # for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
        for future in concurrent.futures.as_completed(futures):
            try:
                # Unpack the results returned by process_year
                year_res, proc_success_res, fips_map_res, hf_success_res = future.result()
                processed_years_summary[year_res] = (proc_success_res, hf_success_res)
                # Collect the FIPS map regardless of success for potential partial aggregation?
                # Or only if proc_success_res is True? Let's collect if it exists.
                if fips_map_res is not None:
                    results_fips_maps_list.append(fips_map_res)
                # Optional: Log success/failure per year as it completes
                # status_str = "SUCCESS" if proc_success_res else "FAILED_PROCESSING"
                # hf_str = f"HF_UPLOAD={'OK' if hf_success_res else 'FAIL/SKIP'}" if proc_success_res else ""
                # print(f"  Year {year_res}: {status_str} {hf_str}")


            except Exception as exc:
                # If a worker crashes, it's hard to know which year failed without extra logic
                print(f'\n!!! A worker process generated an unhandled exception: {exc}')
                traceback.print_exc(limit=2)
                # Mark an unknown year as failed? Or just log it.

    finally:
        print("Shutting down worker processes...")
        executor.shutdown(wait=True) # Ensure all workers finish before proceeding
        print("Workers shut down.")


    # --- Processing Summary ---
    print("\n--- Yearly Processing Summary ---")
    successful_processing_years = sorted([y for y, (p, h) in processed_years_summary.items() if p])
    failed_processing_years = sorted([y for y, (p, h) in processed_years_summary.items() if not p])
    failed_hf_upload_years = sorted([y for y, (p, h) in processed_years_summary.items() if p and not h])

    print(f"Successfully processed and saved locally years: {successful_processing_years}")
    if failed_processing_years:
        print(f"Failed processing/saving years: {failed_processing_years}")
    if failed_hf_upload_years:
        print(f"Successful local save, but failed/skipped Hugging Face upload for years: {failed_hf_upload_years}")


    # --- Memory Cleanup: Full NOAA DF is no longer needed ---
    print("\nCleaning up full NOAA DataFrame from main process memory...")
    del full_noaa_df
    gc.collect()


    # --- Finalization Step (Aggregate and Save/Upload FIPS Map) ---
    finalize_data(
        output_dir=output_dir,
        all_fips_maps=results_fips_maps_list, # Pass the collected maps
        hf_repo_id=hf_repo_id,
        hf_api_instance=hf_api # Pass API instance from main process
    )


    # --- Uploading Results Directory to Kaggle Datasets ---
    print(f"\n--- Uploading results directory to Kaggle Datasets ---")
    if KAGGLE_AVAILABLE and kaggle_username != "your_kaggle_username":
        try:
            print(f"Attempting Kaggle dataset upload/update for handle: {kaggle_handle}")
            print(f"Uploading content from: {output_dir}")
            # Check what files are actually in the directory before upload
            local_files_for_kaggle = list(output_dir.glob('*'))
            if not local_files_for_kaggle:
                 print("Warning: Output directory is empty. Nothing to upload to Kaggle.")
            else:
                print(f"Files found in output directory for Kaggle upload: {[f.name for f in local_files_for_kaggle]}")
                # dataset_upload automatically creates/updates the dataset
                kagglehub.dataset_upload(kaggle_handle, str(output_dir))
                print(f"Kaggle dataset upload/update initiated for handle: {kaggle_handle}")
                print("Note: Check Kaggle UI (kaggle.com/datasets/{kaggle_handle}) for upload progress and status.")
        except Exception as kaggle_err:
            print(f"Error uploading to Kaggle Datasets: {kaggle_err}")
            print("Ensure KAGGLE_USERNAME and KAGGLE_KEY environment variables are correctly set,")
            print("and the kagglehub library is installed and authenticated.")
            traceback.print_exc(limit=2)
    elif not KAGGLE_AVAILABLE:
         print("Skipping Kaggle Dataset upload (kagglehub library not found).")
    else: # Kaggle username is the placeholder
        print("Skipping Kaggle Dataset upload (KAGGLE_USERNAME not set correctly).")


    main_end_time = time.time()
    print("\n" + "="*60)
    print(f"--- Script Execution Finished in {(main_end_time - main_start_time) / 60:.2f} minutes ---")
    print("="*60)

In [ ]:
# Colab Link: https://colab.research.google.com/drive/1sBQv4hDR_akWpaT4DgOB5aFXPgLlrHm3?usp=sharing